# nanochat-ru training notebook (Kaggle T4 x2)

Copy each **code cell** below into a Kaggle Notebook, in order. Markdown cells are for your reference and don't need to be copied.

Before running:
1. Edit `REPO_URL` in Cell 1 to point at your GitHub fork.
2. Set up the Google Drive service account and Kaggle Secrets `GDRIVE_SERVICE_ACCOUNT_JSON` + `GDRIVE_FOLDER_ID` -- see `docs/RCLONE_GDRIVE_SETUP.md`. Attach both secrets to the notebook (Add-ons -> Secrets) before running.
3. Enable a T4 x2 GPU accelerator in Notebook settings, and internet access.

Each session is capped at 12h and can be interrupted earlier. Checkpoints sync to Google Drive continuously (Cell 4) so a killed session loses at most one `--save-every` interval.

## Cell 1: clone repo, install dependencies, Rust toolchain, rclone

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/<YOUR_REPO>.git"  # <-- edit before running
REPO_DIR = "/kaggle/working/repo"
MODEL_TAG = "d4"  # depth=4, ~36.7M params -- see README.md for how this was picked

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

# uv (fast Python package/dependency manager)
if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

# Rust toolchain (needed to build rustbpe, nanochat's tokenizer)
if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

# rclone (for Google Drive checkpoint sync)
if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv sync --extra gpu

print("Cell 1 done.")

## Cell 2: configure rclone from Kaggle Secrets, pull existing checkpoint (if any)

Requires the `GDRIVE_SERVICE_ACCOUNT_JSON` and `GDRIVE_FOLDER_ID` secrets attached to the notebook -- see `docs/RCLONE_GDRIVE_SETUP.md`.

In [ ]:
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
sa_json = secrets.get_secret("GDRIVE_SERVICE_ACCOUNT_JSON")
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID")

sa_path = os.path.expanduser("~/gdrive-sa.json")
with open(sa_path, "w") as f:
    f.write(sa_json)

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"service_account_file = {sa_path}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

# sanity check -- should list existing subfolders (or nothing on a first run), not an auth error
!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

# Pull down anything already on Drive from a previous session: tokenizer + checkpoints
for subdir in ["tokenizer", "base_checkpoints", "chatsft_checkpoints"]:
    remote_path = f"{DRIVE_REMOTE}{subdir}"
    local_path = os.path.join(NANOCHAT_BASE_DIR, subdir)
    listing = subprocess.run(["rclone", "lsf", remote_path], capture_output=True, text=True)
    if listing.returncode == 0 and listing.stdout.strip():
        print(f"Found {subdir} on Drive, downloading...")
        !rclone copy {remote_path} {local_path} --checksum -v
    else:
        print(f"No {subdir} on Drive yet.")

# Figure out whether we can resume base pretraining from a prior session's checkpoint
sys.path.insert(0, REPO_DIR)
from nanochat.checkpoint_manager import find_last_step

RESUME_STEP = -1
base_ckpt_dir = os.path.join(NANOCHAT_BASE_DIR, "base_checkpoints", MODEL_TAG)
if os.path.isdir(base_ckpt_dir):
    try:
        RESUME_STEP = find_last_step(base_ckpt_dir)
        print(f"Found existing base checkpoint at step {RESUME_STEP}, will resume from it.")
    except FileNotFoundError:
        print("base_checkpoints dir exists but has no checkpoints in it yet.")
else:
    print("No prior base checkpoint found, starting fresh.")

os.environ["RESUME_STEP"] = str(RESUME_STEP)

## Cell 3 (optional): reuse cached dataset from Drive instead of re-downloading

The pretraining corpus (ClimbMix parquet shards) is static -- once downloaded, cache it on Drive so future sessions skip the download entirely.

In [ ]:
import os
import subprocess

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = os.environ["NANOCHAT_BASE_DIR"]
DATA_DIR = os.path.join(NANOCHAT_BASE_DIR, "base_data_climbmix")
DATA_REMOTE = f"{DRIVE_REMOTE}base_data_climbmix"

# Chinchilla-style target for the d4 (~36.7M param) model is ~20 tokens/param = ~734M tokens.
# base_train.py's --target-param-data-ratio computes the exact iteration count at train time;
# this just needs to download "enough" shards up front. 20 shards is a safe starting point
# (runs/speedrun.sh uses 8 shards for ~2B chars on a much bigger d24 run); bump this up if
# base_train.py logs that it wants more data than is on disk.
NUM_SHARDS = 20

listing = subprocess.run(["rclone", "lsf", DATA_REMOTE], capture_output=True, text=True)
if listing.returncode == 0 and listing.stdout.strip():
    print("Found cached dataset on Drive, downloading instead of re-fetching from HuggingFace...")
    !rclone copy {DATA_REMOTE} {DATA_DIR} --checksum -v
else:
    print(f"No cached dataset on Drive yet, downloading {NUM_SHARDS} shards from source...")
    !python -m nanochat.dataset -n {NUM_SHARDS}
    print("Caching dataset to Drive for future sessions...")
    !rclone copy {DATA_DIR} {DATA_REMOTE} --checksum -v

## Cell 4: train, with a background watcher syncing new checkpoints to Drive as they're saved

nanochat's `save_checkpoint()` runs in-process inside `base_train.py` and writes straight to local
disk; we don't patch that vendored code to call `rclone` directly. Instead `kaggle/sync_checkpoints.py`
polls the checkpoint dirs every 120s in the background and uploads anything new -- functionally the
same "upload right after every save" behavior, decoupled from nanochat's internals.

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

# depth=4 on T4 x2 trains fast; save often enough that a killed session loses little,
# but not so often that checkpoint I/O and Drive uploads dominate wall-clock time.
SAVE_EVERY = 200

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}, log={SYNC_LOG}")

resume_step = int(os.environ.get("RESUME_STEP", "-1"))
resume_args = f"--resume-from-step={resume_step}" if resume_step >= 0 else ""

# T4 x2: no --fp8 (H100-only feature), modest --device-batch-size to fit 16GB VRAM per GPU.
train_cmd = (
    "torchrun --standalone --nproc_per_node=2 -m scripts.base_train -- "
    "--depth=4 --device-batch-size=16 --target-param-data-ratio=20 "
    f"--save-every={SAVE_EVERY} {resume_args} --run=dummy"
)
print(f"Running: {train_cmd}")
try:
    !{train_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    print("Training cell finished (or was interrupted), sync watcher stopped.")

## Cell 5: final sync + log summary

Run this even if Cell 4 was interrupted (e.g. by the 12h hard limit) -- it catches anything the
background watcher missed since its last poll, so the session can end safely.

In [ ]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
result = subprocess.run(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--once", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print("Final sync exit code:", result.returncode)

if os.path.exists(SYNC_LOG):
    with open(SYNC_LOG) as f:
        lines = f.readlines()
    print("".join(lines[-40:]))

if result.returncode != 0:
    print("WARNING: final sync reported a failure -- check the log above before ending the session.")
else:
    print("All checkpoints synced to Google Drive. Safe to let the session end.")